# Week 2 Day 4 Challenge - Scoring the whole run

## A `score_run` node after the swarm

In the Day 4 lab, the dispatcher fans out to the claim agents with `Send`, but it never comes back together. Nothing in the graph sees the whole run at once, so Part 3 scores each run in cells after the graph has finished.

The challenge: add a `score_run` node after `work_claim` that scores the whole run and puts the scores in the state. LangGraph runs it once, after every claim agent has finished.

This notebook holds only the challenge, with the parts of the lab it needs, so you can run it from the top. Copy it into the `labs` folder before you run it: it reads `../data` and imports `week2day1_payout` from there.

## 3 parts:

PART 1: `score_run` with a stand-in for the claim agent. This costs nothing.

PART 2: `score_run` with the real claim agent, on 5 claims. This costs a few cents.

PART 3: What to remember

## Setup

The cells up to Part 2 call no model and need no keys. They read the harder claims from `data/evals/harder`, the same files as the lab.

In [ ]:
import json
import operator
import time
from datetime import datetime
from pathlib import Path
from typing import Annotated, Literal, TypedDict

import pandas as pd
from langgraph.graph import END, START, StateGraph
from langgraph.types import RetryPolicy, Send


In [ ]:
HARDER = Path("../data/evals/harder")


def load(name):
    """Read one of the JSON files of the harder claims."""
    return json.loads((HARDER / name).read_text(encoding="utf-8"))


claims = {claim["claim_id"]: claim for claim in load("claims.json")}
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../data/policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

### The dispatcher's state

The dispatcher keeps the claims to work and the results that come back. `operator.add` is the reducer on `results`. When several claim agents return results in the same step, LangGraph adds their lists together.

In [ ]:
class Dispatch(TypedDict):
    """What the dispatcher keeps for one run."""

    claims: list[str]
    results: Annotated[list, operator.add]

### Listing the claims and fanning out

`list_claims` picks the claims to work. In the product it will ask the backend for the claims waiting in the queue.

`fan_out` is a conditional edge that returns one `Send` for each claim. A `Send` names the node to run and the input to give it, so `work_claim` runs once for every claim.

In [ ]:
def list_claims(state: Dispatch):
    """The claims to work in this run."""
    return {"claims": list(claims)}


def fan_out(state: Dispatch):
    """Send each claim to a claim agent."""
    return [Send("work_claim", claim_id) for claim_id in state["claims"]]

### Scoring a run

These are the lab's Part 3 measures, copied here so this notebook stands alone.

- `decision_accuracy`, `accuracy`, `cost_of_mistakes` and `straight_through` are Day 1's measures.
- `llm_cost` is what the run's model calls cost.
- `seconds` is the time from the first claim starting to the last claim finishing.
- `slowest_claim` is the time the slowest claim took, and `seconds_per_claim` is the average.

In [ ]:
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL

In [ ]:
def score_table(results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for answer in results:
        expected = labels[answer["claim_id"]]
        rows.append({
            "claim_id": answer["claim_id"],
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "llm_cost": answer["cost"],
            "started": answer["started"],
            "finished": answer["finished"],
            "seconds": (answer["finished"] - answer["started"]).total_seconds(),
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run: quality, cost and time."""
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "llm_cost": table["llm_cost"].sum(),
        "seconds": (table["finished"].max() - table["started"].min()).total_seconds(),
        "slowest_claim": table["seconds"].max(),
        "seconds_per_claim": table["seconds"].mean(),
    }

## Part 1: `score_run` with a stand-in

Part 1's stand-in only recorded times. We build the new node with a stand-in first, so it costs nothing.

### The state and the node

`ScoredDispatch` is the dispatcher's state with a `scores` field. Only `score_run` writes it, so it needs no reducer. `score_run` uses `score_table` and `run_scores`, and turns each score into a plain `float` so the state can be saved.

In [ ]:
class ScoredDispatch(Dispatch):
    """The dispatcher's state, with the scores of the run."""

    scores: dict


def score_run(state: ScoredDispatch):
    """Score the whole run. It runs once, after every claim agent has finished."""
    scores = run_scores(score_table(state["results"]))
    return {"scores": {name: float(value) for name, value in scores.items()}}

### A stand-in that can be scored

Part 1's stand-in only records times. To score a run, each result also needs a decision, a payout, a rationale and a cost. This stand-in answers like the handler on every claim except `CLM-1203`, where it gets the decision wrong on purpose. We know the right scores in advance, so we can check `score_run` against them. It calls no model.

In [ ]:
WRONG_ON_PURPOSE = "CLM-1203"


def scored_stand_in(claim_id):
    """Work a claim for one second, answering like the handler except on one claim."""
    started = datetime.now()
    time.sleep(1)
    expected = labels[claim_id]
    if claim_id == WRONG_ON_PURPOSE:
        decision = "refer" if expected["decision"] != "refer" else "decline"
        answer = {"decision": decision, "payout": 0.0, "rationale": "The stand-in gets this claim wrong"}
    else:
        answer = {"decision": expected["decision"], "payout": expected["payout"], "rationale": "The stand-in copies the handler"}
    return {"results": [{"claim_id": claim_id, **answer, "cost": 0.0, "started": started, "finished": datetime.now()}]}

### The graph

`work_claim` now has an edge to `score_run` instead of `END`. All the `Send`s from `fan_out` run in the same step. A node after them starts in the next step, once every claim has finished and the `results` reducer has joined their lists. So a plain edge is enough for `score_run` to run once, on the whole run.

In [ ]:
def build_scored_dispatcher(work, score=score_run):
    """The dispatcher with `work` as the claim agent and `score` after it."""
    builder = StateGraph(ScoredDispatch)
    builder.add_node("list_claims", list_claims)
    builder.add_node("work_claim", work, retry_policy=RetryPolicy(max_attempts=3))
    builder.add_node("score_run", score)
    builder.add_edge(START, "list_claims")
    builder.add_conditional_edges("list_claims", fan_out, ["work_claim"])
    builder.add_edge("work_claim", "score_run")
    builder.add_edge("score_run", END)
    return builder.compile()


scored_dispatcher = build_scored_dispatcher(scored_stand_in)

In [ ]:
scored_dispatcher

In [ ]:
result = scored_dispatcher.invoke({}, {"max_concurrency": 5})
pd.Series(result["scores"]).round(3)

### Does it run once, after every claim?

A counting wrapper notes how many results `score_run` saw each time it ran. It should run once, on all 25 results, however many claim agents run at once. The stand-in gets one claim wrong on purpose, so the accuracy should be 24 out of 25, which is 0.96.

In [ ]:
calls = []


def counting_score_run(state):
    """score_run, noting how many results it saw each time it ran."""
    calls.append(len(state["results"]))
    return score_run(state)


for at_once in (1, 5, 25):
    calls.clear()
    counted = build_scored_dispatcher(scored_stand_in, counting_score_run)
    out = counted.invoke({}, {"max_concurrency": at_once})
    print(f"{at_once} at once: score_run ran {len(calls)} time(s), on {calls} results. Accuracy {out['scores']['accuracy']:.2f}")

A retried claim stays inside its own step, so a retry doesn't make `score_run` run early or twice. Here CLM-1207 fails once, as before, and `score_run` still runs once on all 25 results.

In [ ]:
tries = {}


def flaky_scored_stand_in(claim_id):
    """The scored stand-in, failing the first time it works CLM-1207."""
    tries[claim_id] = tries.get(claim_id, 0) + 1
    if claim_id == "CLM-1207" and tries[claim_id] == 1:
        raise Exception("429 Too Many Requests")
    return scored_stand_in(claim_id)


calls.clear()
flaky_scored = build_scored_dispatcher(flaky_scored_stand_in, counting_score_run)
out = flaky_scored.invoke({}, {"max_concurrency": 25})
print(f"CLM-1207 took {tries['CLM-1207']} tries. score_run ran {len(calls)} time(s), on {calls} results")

## Part 2: `score_run` with the real claim agent

Now the same graph with the lab's claim agent, the one with the payout tool, on GPT-6 Luna with its reasoning at `low`.

### Connect to OpenRouter and Langfuse

The keys come from the `.env` file in the `undercover` folder. You need `OPENROUTER_API_KEY` and the Langfuse keys from Day 1. If you'd rather not spend anything, stop after Part 1. The lab's free version, `week2day4free.ipynb`, shows how to use a free model.

In [ ]:
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from week2day1_payout import Benefit, Line, Payout, work_out_payout

load_dotenv(override=True)
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
langfuse = get_client()

print("OpenRouter API key found" if openrouter_api_key else "No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")
print("Connected to Langfuse" if langfuse.auth_check() else "No Langfuse keys found. Add them to the .env file")

### The claim agent and its node

These cells are the lab's Part 2, unchanged: the instructions, the tool that records a decision, the payout tool, the agent, and the `work_claim` node that runs it on one claim. Nothing here calls a model yet.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
Use the payout tool to work out the payout on an approved claim, and record the payout it returns.
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [ ]:
@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

In [ ]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)

payout_agent = create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}\n{policy_wording}\n")

In [ ]:
def work_claim(claim_id):
    """Run the claim agent on one claim and return its result."""
    started = datetime.now()
    result = payout_agent.invoke({"messages": [{"role": "user", "content": make_user_prompt(claims[claim_id])}]})
    answer = recorded.get(claim_id, {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    print(f"{claim_id}: {answer['decision']} ${answer['payout']:,.2f}")
    return {"results": [{"claim_id": claim_id, **answer, "cost": cost, "started": started, "finished": datetime.now()}]}

In [ ]:
def send_scores(trace_id, scores):
    """Add a run's scores to its trace in Langfuse."""
    for name, value in scores.items():
        langfuse.create_score(trace_id=trace_id, name=name, value=float(value))

### Five claims, scored inside the graph

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td>
            <h3 style="color:#EF476F;">This cell calls GPT-6 Luna</h3>
            <span style="color:#EF476F;">It works 5 claims and costs a few cents. Every cell above it is free.</span>
        </td>
    </tr>
</table>

The same graph with the real `work_claim` in place of the stand-in. The next cell keeps the 5 claims marked `tiny` in the labels, as the lab's `TINY` switch does. The scores come back in the state, and we check them against the scoring outside the graph.

In [ ]:
langfuse_handler = CallbackHandler()

claims = {claim_id: claim for claim_id, claim in claims.items() if labels[claim_id]["tiny"]}
print(f"The swarm works {len(claims)} claims")

recorded.clear()
real_scored = build_scored_dispatcher(work_claim)
result = real_scored.invoke({}, {"max_concurrency": 5, "callbacks": [langfuse_handler], "run_name": "Scored swarm of 5"})
pd.Series(result["scores"]).round(3)

In [ ]:
outside = {name: float(value) for name, value in run_scores(score_table(result["results"])).items()}
print("Same as scoring outside the graph:", outside == result["scores"])

In [ ]:
send_scores(langfuse_handler.last_trace_id, result["scores"])

The scores are on the `Scored swarm of 5` trace in Langfuse, next to the run's timeline. The scores in the state are the same numbers the lab's Part 3 worked out after the run. Now the graph does that work itself.

## Part 3: What to remember

- All the `Send`s from `fan_out` run in one step, and a node after them starts in the next step. That makes a plain edge from `work_claim` to `score_run` a join: it runs once, after every claim agent has finished.
- The `results` reducer, `operator.add`, has already put every claim's result in one list by then, so `score_run` sees the whole run.
- A retried claim stays inside its own step. A retry doesn't make `score_run` run early or twice.
- With no claims, the fan-out sends nothing, so `score_run` doesn't run and the state has no `scores`.
- `score_run` returns plain floats, so the state can be saved by a checkpointer.

In UnderCover's product, as the lab's note says, each claim keeps its own thread and the dispatcher only starts them. So the single graph state that `score_run` joins is how the lab works, not how the product will.